# 06 · Falcon 9 Launch Geography | Interactive Folium

**Question:** How do historical Falcon 9 recovery outcomes vary across launch sites?  
**Source:** `data/processed/falcon9_labeled_historical.csv` from Notebook 02 (IBM course historical snapshot, approximately 2010–2020).  
**Deliverables:** `reports/falcon9_launch_sites_map.html` and `data/processed/falcon9_site_summary.csv`.

This is a portfolio reconstruction of the IBM capstone Folium lab. Original coursework is preserved in `original-coursework/`. **The outcome represents first-stage recovery, not mission success.**

**Interpretation caveat:** A launch-site association is not causal evidence; booster generations, launch dates, orbit, payload and operational choices differ between sites. The dataset is historical, not a current SpaceX operational feed.

In [1]:
from pathlib import Path
import pandas as pd
import folium
from folium.plugins import MarkerCluster, Fullscreen, MousePosition
from IPython.display import display

ROOT = Path.cwd()
SOURCE = ROOT / 'data' / 'processed' / 'falcon9_labeled_historical.csv'
REPORTS = ROOT / 'reports'
PROCESSED = ROOT / 'data' / 'processed'
REPORTS.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)
if not SOURCE.exists():
    raise FileNotFoundError(f'Run Notebook 02 first: {SOURCE}')
df = pd.read_csv(SOURCE)
required = {'LaunchSite', 'Latitude', 'Longitude', 'Class', 'Date'}
if missing := required - set(df.columns):
    raise ValueError(f'Columns missing: {sorted(missing)}')
for col in ('Latitude', 'Longitude', 'Class'):
    df[col] = pd.to_numeric(df[col], errors='coerce')
df['Date'] = pd.to_datetime(df['Date'], errors='raise')
if not df['Class'].dropna().isin([0,1]).all():
    raise ValueError('Recovery class must be 0, 1 or missing')
if df['Class'].isna().any():
    raise ValueError('Missing recovery labels: resolve in Notebook 02')
if (df['Latitude'].notna() & ~df['Latitude'].between(-90,90)).any() or (df['Longitude'].notna() & ~df['Longitude'].between(-180,180)).any():
    raise ValueError('Invalid geographical coordinates')
print('Loaded',len(df),'historical launches; coordinate coverage:',int(df[['Latitude','Longitude']].notna().all(axis=1).sum()))
display(df.head(3))

Loaded 90 historical launches; coordinate coverage: 90


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude,Class
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857,0
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857,0
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857,0


## 1 · Launch-site geography and coverage

Each launch site is represented by its historical coordinates. Missing coordinates are **reported**, not silently fabricated. Site-level recovery rates are calculated from launches with known binary outcomes.

In [2]:
geo = df.dropna(subset=['Latitude','Longitude']).copy()
missing_geo = df[df[['Latitude','Longitude']].isna().any(axis=1)]
if geo.empty:
    raise ValueError('No launch coordinates available')
print('Excluded records with missing coordinates:',len(missing_geo))
site_summary = (geo.groupby('LaunchSite',dropna=False)
    .agg(Launches=('Class','size'),Recoveries=('Class','sum'),RecoveryRate=('Class','mean'),
         Latitude=('Latitude','median'),Longitude=('Longitude','median'),
         FirstLaunch=('Date','min'),LastLaunch=('Date','max'))
    .reset_index().sort_values('Launches',ascending=False))
site_summary['Failures'] = site_summary['Launches']-site_summary['Recoveries']
site_summary['FirstLaunch'] = site_summary['FirstLaunch'].dt.strftime('%Y-%m-%d')
site_summary['LastLaunch'] = site_summary['LastLaunch'].dt.strftime('%Y-%m-%d')
display(site_summary)
site_summary.to_csv(PROCESSED/'falcon9_site_summary.csv',index=False)

Excluded records with missing coordinates: 0


,LaunchSite,Launches,Recoveries,RecoveryRate,Latitude,Longitude,FirstLaunch,LastLaunch,Failures
0,CCAFS SLC 40,55,33,0.600000,28.561857,-80.577366,2010-06-04,2020-11-05,22
1,KSC LC 39A,22,17,0.772727,28.608058,-80.603956,2017-02-19,2020-10-18,5
2,VAFB SLC 4E,13,10,0.769231,34.632093,-120.610829,2013-09-29,2019-06-12,3


## 2 · Interactive site map

Circle size indicates launch volume, while colour encodes historical first-stage recovery rate. The popup displays the sample size to prevent misleading comparisons.

In [3]:
center=[float(site_summary['Latitude'].mean()),float(site_summary['Longitude'].mean())]
site_map=folium.Map(location=center,zoom_start=4,tiles='OpenStreetMap',control_scale=True)
for row in site_summary.itertuples():
    rate=float(row.RecoveryRate)
    color='#16a34a' if rate>=0.75 else '#f59e0b' if rate>=0.5 else '#dc2626'
    popup=(f'<b>{row.LaunchSite}</b><br>Historical launches: {row.Launches}'
           f'<br>Recovered: {int(row.Recoveries)} / {row.Launches}'
           f'<br>Recovery rate: {rate:.1%}<br>Period: {row.FirstLaunch} to {row.LastLaunch}')
    folium.CircleMarker(location=[row.Latitude,row.Longitude],radius=min(26,8+row.Launches**0.5*2),
        color=color,weight=2,fill=True,fill_color=color,fill_opacity=0.75,
        tooltip=f'{row.LaunchSite}: {rate:.0%} (n={row.Launches})',popup=folium.Popup(popup,max_width=320)).add_to(site_map)
Fullscreen().add_to(site_map)
MousePosition(position='bottomright',separator=' | ',prefix='Lat/Lon').add_to(site_map)

from IPython.display import display, FileLink

site_map.save("reports/falcon9_launch_sites_map.html")
display(FileLink("reports/falcon9_launch_sites_map.html"))

c:\Users\kanwa\Downloads\New folder\notebooks\reports\falcon9_launch_sites_map.html

## 3 · Launch outcomes by site

Multiple launches share identical pad coordinates, so plotting all individual markers at the exact same point obscures observations. This **cluster layer** allows inspection of individual launch records while retaining a readable site-level overview.

In [4]:
detail_map=folium.Map(location=center,zoom_start=4,tiles='OpenStreetMap',control_scale=True)
cluster=MarkerCluster(name='Historical launches').add_to(detail_map)
for row in geo.itertuples():
    success=int(row.Class)==1
    label='Recovered' if success else 'Not recovered'
    popup=f'<b>{row.LaunchSite}</b><br>{row.Date.date()}<br>Flight: {getattr(row,"FlightNumber","n/a")}<br>First-stage recovery: {label}'
    folium.Marker(location=[row.Latitude,row.Longitude],tooltip=label,
        icon=folium.Icon(color='green' if success else 'red',icon='info-sign'),
        popup=folium.Popup(popup,max_width=300)).add_to(cluster)
Fullscreen().add_to(detail_map)

detail_map.save("reports/falcon9_launch_outcomes_map.html")
display(FileLink("reports/falcon9_launch_outcomes_map.html"))


c:\Users\kanwa\Downloads\New folder\notebooks\reports\falcon9_launch_outcomes_map.html

## 4 · Export and interpretation

We export both maps as self-contained HTML files. Base-map tiles may still require internet access. We deliberately **do not** infer distances to highways, coastlines or cities without independently sourced coordinates: the original coursework used manually selected map points, not a validated nearest-feature geospatial calculation.

In [5]:
site_file=REPORTS/'falcon9_launch_sites_map.html'
detail_file=REPORTS/'falcon9_launch_outcomes_map.html'
site_map.save(str(site_file)); detail_map.save(str(detail_file))
assert site_file.exists() and detail_file.exists()
print('Saved:',site_file,'and',detail_file)
print('Saved:',PROCESSED/'falcon9_site_summary.csv')

Saved: c:\Users\kanwa\Downloads\New folder\notebooks\reports\falcon9_launch_sites_map.html and c:\Users\kanwa\Downloads\New folder\notebooks\reports\falcon9_launch_outcomes_map.html
Saved: c:\Users\kanwa\Downloads\New folder\notebooks\data\processed\falcon9_site_summary.csv


### Analytical limits
- The labels come from the IBM course recovery-outcome mapping, including its treatment of no-attempt outcomes.
- Site-level comparisons are descriptive, not controlled for mission mix or changes over time.
- Multiple records at identical coordinates are aggregated or clustered, not geospatially jittered into fictitious positions.
- Maps show launch locations, **not** booster landing locations.

**Next:** Notebook 07's Dash app uses the same processed dataset for filterable operational-style analytics.